In [43]:
%reload_ext autoreload
%autoreload 2

from comet_ml import Experiment
import h5py
import matplotlib.pyplot as plt
import numpy as np
import argparse


import importlib
import random
import os
from algorithms.server.server import Server
from algorithms.trainmodel.models import *
from utils.plot_utils import *
import torch
#torch.manual_seed(0)
seeds = [0, 10, 20, 30, 40]
def main(experiment, dataset, algorithm, model, batch_size, learning_rate, alpha, eta, L, rho, num_glob_iters,
         local_epochs, optimizer, numedges, times, commet, gpu):

    device = torch.device("cuda:{}".format(gpu) if torch.cuda.is_available() and gpu != -1 else "cpu")

    for i in range(times):
        print("---------------Running time:------------",i)
        torch.manual_seed(seeds[i])

        # Generate model
        if(model == "mclr"):
            if(dataset == "human_activity"):
                model = Mclr_CrossEntropy(561,6).to(device), model
                print("Hey")
            else:
                model = Mclr_Logistic().to(device), model

        if(model == "linear_regression"):
            model = Linear_Regression(40,1).to(device), model

        if model == "logistic_regression":
            model = Logistic_Regression(300).to(device), model
        
        if model == "MLP" and dataset == "a9a":
            model = DNN( input_dim = 123, output_dim = 2).to(device), model
        if model == "MLP" and dataset == "human_activity":
            model = DNN( input_dim = 561, mid_dim = 561, output_dim = 6).to(device), model
        if model == "MLP" and dataset == "w8a":
            model = DNN( input_dim = 300, output_dim = 2).to(device), model
        if model == "MLP" and dataset == "Mnist":
            model = DNN().to(device), model
        if model == 'CNN':
            model = Net().to(device), model
        # select algorithm
        if(commet):
            experiment.set_name(dataset + "_" + algorithm + "_" + model[1] + "_" + str(batch_size) + "b_" + str(learning_rate) + "lr_" + str(alpha) + "al_" + str(eta) + "eta_" + str(L) + "L_" + str(rho) + "p_" +  str(num_glob_iters) + "ge_"+ str(local_epochs) + "le_"+ str(numedges) +"u")
        server = Server(experiment, device, dataset, algorithm, model, batch_size, learning_rate, alpha, eta,  L, num_glob_iters, local_epochs, optimizer, numedges, i)
        
        server.train()
        server.test()


In [44]:
sophia_params = {
    "dataset": "Mnist",
    "algorithm": "Sophia",
    "model": "MLP",
    "batch_size": 64,
    "learning_rate": 0.01,
    "alpha": (0.90, 0.95),
    "eta": 1.0 * 5,
    "L": 0.1,
    "rho": 20,
    "num_glob_iters": 2 * 41,
    "local_epochs": 1,
    "optimizer": "Sophia",
    "numedges": 32,
    "times": 5,
    "commet": False,
    "gpu": 0
}



DONE_params = {
    "dataset": "Mnist",
    "algorithm": "DONE",
    "model": "MLP",
    "batch_size": 0,
    "learning_rate": 1,
    "alpha": 0.004,
    "eta": 1.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 41,
    "local_epochs": 80,
    "optimizer": "DONE",
    "numedges": 32,
    "times": 5,
    "commet": False,
    "gpu": 0
}

GD_params = {
    "dataset": "Mnist",
    "algorithm": "FedAvg",
    "model": "MLP",
    "batch_size": 64,
    "learning_rate": 0.008,
    "alpha": 0.03,
    "eta": 20.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 2 * 41,
    "local_epochs": 10,
    "optimizer": "DONE",
    "numedges": 32,
    "times": 5,
    "commet": False,
    "gpu": 0
}

'''
Newton_params = {
    "dataset": "Mnist",
    "algorithm": "Newton",
    "model": "MLP",
    "batch_size": 0,
    "learning_rate": 1,
    "alpha": 0.03,
    "eta": 1.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 500,
    "local_epochs": 40,
    "optimizer": "Newton",
    "numedges": 30,
    "times": 1,
    "commet": True,
    "gpu": 0
}

'''

'\nNewton_params = {\n    "dataset": "Mnist",\n    "algorithm": "Newton",\n    "model": "MLP",\n    "batch_size": 0,\n    "learning_rate": 1,\n    "alpha": 0.03,\n    "eta": 1.0,\n    "L": 1e-3,\n    "rho": 0.01,\n    "num_glob_iters": 500,\n    "local_epochs": 40,\n    "optimizer": "Newton",\n    "numedges": 30,\n    "times": 1,\n    "commet": True,\n    "gpu": 0\n}\n\n'

In [45]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **sophia_params)
experiment.end()    


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     url                   : https://www.comet.com/abdulmomen96/sophia/f47e0bce5dd045a8a5dabba20c0ec472
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     glob_acc [400]   : (0.07769074557586307, 0.9671018276762402)
COMET INFO:     loss [1564]      : (0.00046353929792530835, 2.3385190963745117)
COMET INFO:     train_acc [400]  : (0.0761770171906458, 0.9998064116462754)
COMET INFO:     train_loss [400] : (0.0038645565463641084, 2.3219452967709464)
COMET INFO:   Uploads:
COMET INFO:     conda-environment-definition : 1
COMET INFO:     conda-info                   : 1
COMET INFO:     conda-specification          : 1
COMET INFO:     environment details          : 1
COMET INFO

---------------Running time:------------ 0
Number of edges / total edges: 32  /  32
Total number of parameters: 79510
-------------Round number:  0  -------------
Win rate = 0.0817507231794743
-------------Round number:  1  -------------
Average Test Accuracy          :  0.07769074557586307
Average Global Trainning Accuracy:  0.0761770171906458
Average Global Trainning Loss    :  2.3219452967709464
Win rate = 0.1395925040875362
-------------Round number:  2  -------------
Average Test Accuracy          :  0.6948651000870322
Average Global Trainning Accuracy:  0.6916911878581384
Average Global Trainning Loss    :  1.2920876205087501
Win rate = 0.14183121619922023
-------------Round number:  3  -------------
Average Test Accuracy          :  0.7366405570060922
Average Global Trainning Accuracy:  0.7346097258788912
Average Global Trainning Loss    :  0.8315141573582934
Win rate = 0.14608225380455286
-------------Round number:  4  -------------
Average Test Accuracy          :  0.798027270

Win rate = 0.5995220726952585
-------------Round number:  35  -------------
Average Test Accuracy          :  0.9559617058311576
Average Global Trainning Accuracy:  0.9778728511692737
Average Global Trainning Loss    :  0.0714943956549699
Win rate = 0.6188152433656144
-------------Round number:  36  -------------
Average Test Accuracy          :  0.9561937917029301
Average Global Trainning Accuracy:  0.9795377110113056
Average Global Trainning Loss    :  0.06605576338996196
Win rate = 0.6373412149415167
-------------Round number:  37  -------------
Average Test Accuracy          :  0.9575863069335654
Average Global Trainning Accuracy:  0.9808734706520056
Average Global Trainning Loss    :  0.06095923804380179
Win rate = 0.6582065149037857
-------------Round number:  38  -------------
Average Test Accuracy          :  0.9583986074847693
Average Global Trainning Accuracy:  0.9827319188477621
Average Global Trainning Loss    :  0.05596438911985055
Win rate = 0.680329518299585
------------

Average Test Accuracy          :  0.9661154627212069
Average Global Trainning Accuracy:  0.9995353879510609
Average Global Trainning Loss    :  0.006362250996980022
Win rate = 0.9584957866934977
-------------Round number:  70  -------------
Average Test Accuracy          :  0.9663475485929794
Average Global Trainning Accuracy:  0.9995741056218058
Average Global Trainning Loss    :  0.006047520453630447
Win rate = 0.9596528738523457
-------------Round number:  71  -------------
Average Test Accuracy          :  0.9661154627212069
Average Global Trainning Accuracy:  0.9996321821279232
Average Global Trainning Loss    :  0.005770854763853969
Win rate = 0.9602565715004402
-------------Round number:  72  -------------
Average Test Accuracy          :  0.9659994197853206
Average Global Trainning Accuracy:  0.9996321821279232
Average Global Trainning Loss    :  0.005513913014438364
Win rate = 0.9605332662558169
-------------Round number:  73  -------------
Average Test Accuracy          :  0.

Average Test Accuracy          :  0.9337975050768784
Average Global Trainning Accuracy:  0.9465889732073719
Average Global Trainning Loss    :  0.20176666765477388
Win rate = 0.3743176958873098
-------------Round number:  23  -------------
Average Test Accuracy          :  0.9388453727879316
Average Global Trainning Accuracy:  0.95160291156884
Average Global Trainning Loss    :  0.18304077545444866
Win rate = 0.3922022387121117
-------------Round number:  24  -------------
Average Test Accuracy          :  0.9422106179286336
Average Global Trainning Accuracy:  0.9563458262350937
Average Global Trainning Loss    :  0.1655506348488075
Win rate = 0.4069676770217583
-------------Round number:  25  -------------
Average Test Accuracy          :  0.9423266608645199
Average Global Trainning Accuracy:  0.9590754220226111
Average Global Trainning Loss    :  0.15190369569241716
Win rate = 0.42050056596654506
-------------Round number:  26  -------------
Average Test Accuracy          :  0.941804

...

Average Test Accuracy          :  0.9541630403249203
Average Global Trainning Accuracy:  0.9759756853027722
Average Global Trainning Loss    :  0.0821277606001723
Win rate = 0.5831845050936989
-------------Round number:  34  -------------
Average Test Accuracy          :  0.9541630403249203
Average Global Trainning Accuracy:  0.9768468328945331
Average Global Trainning Loss    :  0.07675460146661327
Win rate = 0.6053703936611747
-------------Round number:  35  -------------
Average Test Accuracy          :  0.9550913838120104
Average Global Trainning Accuracy:  0.9781632336998606
Average Global Trainning Loss    :  0.0714675409424244
Win rate = 0.6287007923531631
-------------Round number:  36  -------------
Average Test Accuracy          :  0.956135770234987
Average Global Trainning Accuracy:  0.9787439987610346
Average Global Trainning Loss    :  0.06686698650084211
Win rate = 0.642900264117721
-------------Round number:  37  -------------
Average Test Accuracy          :  0.95735422

Average Test Accuracy          :  0.9664055700609225
Average Global Trainning Accuracy:  0.9994192349388261
Average Global Trainning Loss    :  0.007073806944657022
Win rate = 0.9564205760281725
-------------Round number:  69  -------------
Average Test Accuracy          :  0.9664635915288656
Average Global Trainning Accuracy:  0.9994192349388261
Average Global Trainning Loss    :  0.006711172618159979
Win rate = 0.9577537416677148
-------------Round number:  70  -------------
Average Test Accuracy          :  0.9666376559326951
Average Global Trainning Accuracy:  0.9994773114449435
Average Global Trainning Loss    :  0.006380805183494536
Win rate = 0.9593007168909571
-------------Round number:  71  -------------
Average Test Accuracy          :  0.9666376559326951
Average Global Trainning Accuracy:  0.9995160291156884
Average Global Trainning Loss    :  0.0060819248047631206
Win rate = 0.9593636020626336
-------------Round number:  72  -------------
Average Test Accuracy          :  0

Win rate = 0.3400452773236071
-------------Round number:  21  -------------
Average Test Accuracy          :  0.9342036553524804
Average Global Trainning Accuracy:  0.9482151153786588
Average Global Trainning Loss    :  0.20402405153805947
Win rate = 0.35957741164633383
-------------Round number:  22  -------------
Average Test Accuracy          :  0.9380330722367276
Average Global Trainning Accuracy:  0.9514867585566052
Average Global Trainning Loss    :  0.18882349022257822
Win rate = 0.3756005533895107
-------------Round number:  23  -------------
Average Test Accuracy          :  0.9405860168262257
Average Global Trainning Accuracy:  0.9547390428991792
Average Global Trainning Loss    :  0.1749019164944537
Win rate = 0.38890705571626216
-------------Round number:  24  -------------
Average Test Accuracy          :  0.9408761241659414
Average Global Trainning Accuracy:  0.956442620411956
Average Global Trainning Loss    :  0.1665033201915073
Win rate = 0.40778518425355303
----------

Average Test Accuracy          :  0.9645488830867421
Average Global Trainning Accuracy:  0.9967089979866811
Average Global Trainning Loss    :  0.015450366384864536
Win rate = 0.9169286882153188
-------------Round number:  56  -------------
Average Test Accuracy          :  0.9646069045546852
Average Global Trainning Accuracy:  0.9970574570233854
Average Global Trainning Loss    :  0.014340905669083697
Win rate = 0.9194692491510502
-------------Round number:  57  -------------
Average Test Accuracy          :  0.9647809689585146
Average Global Trainning Accuracy:  0.9974059160600898
Average Global Trainning Loss    :  0.01350847102178958
Win rate = 0.9255188026663312
-------------Round number:  58  -------------
Average Test Accuracy          :  0.9654192051058892
Average Global Trainning Accuracy:  0.9978318104382841
Average Global Trainning Loss    :  0.012537198408628112
Win rate = 0.9313797006665828
-------------Round number:  59  -------------
Average Test Accuracy          :  0.9

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     url                   : https://www.comet.com/abdulmomen96/sophia/872fd8488fd4498fb46dd150d401bb74
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     glob_acc [405]   : (0.07769074557586307, 0.9672178706121265)
COMET INFO:     loss [1584]      : (0.0004824207862839103, 2.3385188579559326)
COMET INFO:     train_acc [405]  : (0.0761770171906458, 0.9998451293170203)
COMET INFO:     train_loss [405] : (0.0037077265256917455, 2.3219452967709464)
COMET INFO:   Uploads:
COMET INFO:     conda-environment-definition : 1
COMET INFO:     conda-info                   : 1
COMET INFO:     conda-specification          : 1
COMET INFO:     environment details          : 1
COMET INFO:

In [46]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **DONE_params)
experiment.end()

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/abdulmomen96/sophia/2f26565094a44f35ac0adaa1d4e0280d



---------------Running time:------------ 0
Number of edges / total edges: 32  /  32
-------------Round number:  0  -------------
Average Test Accuracy          :  0.0777487670438062
Average Global Trainning Accuracy:  0.07615765835527334
Average Global Trainning Loss    :  2.3220207659807186
-------------Round number:  1  -------------
Average Test Accuracy          :  0.651348999129678
Average Global Trainning Accuracy:  0.6479595787517423
Average Global Trainning Loss    :  1.406780250600124
-------------Round number:  2  -------------
Average Test Accuracy          :  0.7727299100667246
Average Global Trainning Accuracy:  0.7687587114759176
Average Global Trainning Loss    :  0.8672344602998684
-------------Round number:  3  -------------
Average Test Accuracy          :  0.8387583405860168
Average Global Trainning Accuracy:  0.8399024314697228
Average Global Trainning Loss    :  0.6182945077774121
-------------Round number:  4  -------------
Average Test Accuracy          :  0.8630

-------------Round number:  39  -------------
Average Test Accuracy          :  0.9430229184798375
Average Global Trainning Accuracy:  0.9507317639770791
Average Global Trainning Loss    :  0.17049082436575616
-------------Round number:  40  -------------
Average Test Accuracy          :  0.9435451116913258
Average Global Trainning Accuracy:  0.9516222704042124
Average Global Trainning Loss    :  0.16850960636833087
---------------Running time:------------ 1
Number of edges / total edges: 32  /  32
-------------Round number:  0  -------------
Average Test Accuracy          :  0.0777487670438062
Average Global Trainning Accuracy:  0.07615765835527334
Average Global Trainning Loss    :  2.3220206147398175
-------------Round number:  1  -------------
Average Test Accuracy          :  0.651348999129678
Average Global Trainning Accuracy:  0.6479595787517423
Average Global Trainning Loss    :  1.4067800993592225
-------------Round number:  2  -------------
Average Test Accuracy          :  0

-------------Round number:  37  -------------
Average Test Accuracy          :  0.9416304032492022
Average Global Trainning Accuracy:  0.9492798513241444
Average Global Trainning Loss    :  0.17464541192576855
-------------Round number:  38  -------------
Average Test Accuracy          :  0.9425007252683493
Average Global Trainning Accuracy:  0.949976769397553
Average Global Trainning Loss    :  0.1725352232497193
-------------Round number:  39  -------------
Average Test Accuracy          :  0.9430229184798375
Average Global Trainning Accuracy:  0.9507317639770791
Average Global Trainning Loss    :  0.17049084327086883
-------------Round number:  40  -------------
Average Test Accuracy          :  0.9435451116913258
Average Global Trainning Accuracy:  0.9516222704042124
Average Global Trainning Loss    :  0.1685095874632182
---------------Running time:------------ 2
Number of edges / total edges: 32  /  32
-------------Round number:  0  -------------
Average Test Accuracy          :  

-------------Round number:  35  -------------
Average Test Accuracy          :  0.9408181026979983
Average Global Trainning Accuracy:  0.9477892210004646
Average Global Trainning Loss    :  0.17909297201680346
-------------Round number:  36  -------------
Average Test Accuracy          :  0.9413402959094865
Average Global Trainning Accuracy:  0.9483506272262661
Average Global Trainning Loss    :  0.17682984097926668
-------------Round number:  37  -------------
Average Test Accuracy          :  0.9416304032492022
Average Global Trainning Accuracy:  0.9492798513241444
Average Global Trainning Loss    :  0.17464543083088122
-------------Round number:  38  -------------
Average Test Accuracy          :  0.9425007252683493
Average Global Trainning Accuracy:  0.949976769397553
Average Global Trainning Loss    :  0.1725352232497193
-------------Round number:  39  -------------
Average Test Accuracy          :  0.9430229184798375
Average Global Trainning Accuracy:  0.9507317639770791
Average 

-------------Round number:  33  -------------
Average Test Accuracy          :  0.9399477806788512
Average Global Trainning Accuracy:  0.9464921790305095
Average Global Trainning Loss    :  0.1838648304105525
-------------Round number:  34  -------------
Average Test Accuracy          :  0.9402378880185669
Average Global Trainning Accuracy:  0.9472471736100356
Average Global Trainning Loss    :  0.18143714927234977
-------------Round number:  35  -------------
Average Test Accuracy          :  0.9408181026979983
Average Global Trainning Accuracy:  0.9477892210004646
Average Global Trainning Loss    :  0.17909297201680346
-------------Round number:  36  -------------
Average Test Accuracy          :  0.9413402959094865
Average Global Trainning Accuracy:  0.9483506272262661
Average Global Trainning Loss    :  0.17682982207415401
-------------Round number:  37  -------------
Average Test Accuracy          :  0.9416304032492022
Average Global Trainning Accuracy:  0.9492798513241444
Average

-------------Round number:  31  -------------
Average Test Accuracy          :  0.9389033942558747
Average Global Trainning Accuracy:  0.9450402663775748
Average Global Trainning Loss    :  0.18900628375696918
-------------Round number:  32  -------------
Average Test Accuracy          :  0.9394255874673629
Average Global Trainning Accuracy:  0.945679107944866
Average Global Trainning Loss    :  0.18639015645568763
-------------Round number:  33  -------------
Average Test Accuracy          :  0.9399477806788512
Average Global Trainning Accuracy:  0.9464921790305095
Average Global Trainning Loss    :  0.1838648304105525
-------------Round number:  34  -------------
Average Test Accuracy          :  0.9402378880185669
Average Global Trainning Accuracy:  0.9472471736100356
Average Global Trainning Loss    :  0.18143716817746244
-------------Round number:  35  -------------
Average Test Accuracy          :  0.9408181026979983
Average Global Trainning Accuracy:  0.9477892210004646
Average 

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     url                   : https://www.comet.com/abdulmomen96/sophia/2f26565094a44f35ac0adaa1d4e0280d
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     glob_acc [205]   : (0.0777487670438062, 0.9435451116913258)
COMET INFO:     loss [656]       : (0.09839005768299103, 2.3568267822265625)
COMET INFO:     train_acc [205]  : (0.07615765835527334, 0.9516222704042124)
COMET INFO:     train_loss [205] : (0.1685095874632182, 2.3220207659807186)
COMET INFO:   Uploads:
COMET INFO:     conda-environment-definition : 1
COMET INFO:     conda-info                   : 1
COMET INFO:     conda-specification          : 1
COMET INFO:     environment details          : 1
COMET INFO:     

In [47]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **GD_params)
experiment.end()

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/abdulmomen96/sophia/90c6b2efb6804e06b26bfb5389728792



---------------Running time:------------ 0
Number of edges / total edges: 32  /  32
-------------Round number:  0  -------------
Average Test Accuracy          :  0.0777487670438062
Average Global Trainning Accuracy:  0.07615765835527334
Average Global Trainning Loss    :  2.3220207659807186
-------------Round number:  1  -------------
Average Test Accuracy          :  0.1456338845372788
Average Global Trainning Accuracy:  0.14313922874399876
Average Global Trainning Loss    :  2.254065204187316
-------------Round number:  2  -------------
Average Test Accuracy          :  0.2615027560197273
Average Global Trainning Accuracy:  0.26283490785194363
Average Global Trainning Loss    :  2.1880913519242684
-------------Round number:  3  -------------
Average Test Accuracy          :  0.3791702930084131
Average Global Trainning Accuracy:  0.3796074028186464
Average Global Trainning Loss    :  2.1239544716489855
-------------Round number:  4  -------------
Average Test Accuracy          :  0.4

-------------Round number:  39  -------------
Average Test Accuracy          :  0.8369016536118363
Average Global Trainning Accuracy:  0.8362629704196995
Average Global Trainning Loss    :  0.7082428408606938
-------------Round number:  40  -------------
Average Test Accuracy          :  0.8388743835219031
Average Global Trainning Accuracy:  0.8387989778534923
Average Global Trainning Loss    :  0.6937992591615688
-------------Round number:  41  -------------
Average Test Accuracy          :  0.8424717145343777
Average Global Trainning Accuracy:  0.8416253678178721
Average Global Trainning Loss    :  0.6799986781545222
-------------Round number:  42  -------------
Average Test Accuracy          :  0.845082680591819
Average Global Trainning Accuracy:  0.8441613752516649
Average Global Trainning Loss    :  0.6669559653041273
-------------Round number:  43  -------------
Average Test Accuracy          :  0.8475195822454308
Average Global Trainning Accuracy:  0.8463682824841258
Average Glo

-------------Round number:  79  -------------
Average Test Accuracy          :  0.8886568030171164
Average Global Trainning Accuracy:  0.8882801610655103
Average Global Trainning Loss    :  0.42569716005885083
-------------Round number:  80  -------------
Average Test Accuracy          :  0.8892370176965477
Average Global Trainning Accuracy:  0.8888028496205669
Average Global Trainning Loss    :  0.42248495455513396
-------------Round number:  81  -------------
Average Test Accuracy          :  0.8899332753118654
Average Global Trainning Accuracy:  0.8895965618708378
Average Global Trainning Loss    :  0.41935872950383307
---------------Running time:------------ 1
Number of edges / total edges: 32  /  32
-------------Round number:  0  -------------
Average Test Accuracy          :  0.0777487670438062
Average Global Trainning Accuracy:  0.07615765835527334
Average Global Trainning Loss    :  2.3220206147398175
-------------Round number:  1  -------------
Average Test Accuracy          :

-------------Round number:  36  -------------
Average Test Accuracy          :  0.8281984334203656
Average Global Trainning Accuracy:  0.826932011770172
Average Global Trainning Loss    :  0.7561567902324996
-------------Round number:  37  -------------
Average Test Accuracy          :  0.8307513780098637
Average Global Trainning Accuracy:  0.830358525631098
Average Global Trainning Loss    :  0.7394655418780006
-------------Round number:  38  -------------
Average Test Accuracy          :  0.8337684943429069
Average Global Trainning Accuracy:  0.8332236332662227
Average Global Trainning Loss    :  0.7233607301184761
-------------Round number:  39  -------------
Average Test Accuracy          :  0.8368436321438932
Average Global Trainning Accuracy:  0.8361274585720923
Average Global Trainning Loss    :  0.708341525548823
-------------Round number:  40  -------------
Average Test Accuracy          :  0.8390484479257325
Average Global Trainning Accuracy:  0.8384892364875329
Average Globa

...

-------------Round number:  29  -------------
Average Test Accuracy          :  0.8040615027560197
Average Global Trainning Accuracy:  0.7997134892364876
Average Global Trainning Loss    :  0.9013289235519591
-------------Round number:  30  -------------
Average Test Accuracy          :  0.8086451987235277
Average Global Trainning Accuracy:  0.8043983273966239
Average Global Trainning Loss    :  0.8772183259205126
-------------Round number:  31  -------------
Average Test Accuracy          :  0.8127647229474906
Average Global Trainning Accuracy:  0.8090250890506427
Average Global Trainning Loss    :  0.8543382242624283
-------------Round number:  32  -------------
Average Test Accuracy          :  0.8157238178125906
Average Global Trainning Accuracy:  0.8132259563264674
Average Global Trainning Loss    :  0.8327114559538098
-------------Round number:  33  -------------
Average Test Accuracy          :  0.8176965477226574
Average Global Trainning Accuracy:  0.816710546693511
Average Glo

-------------Round number:  69  -------------
Average Test Accuracy          :  0.8824485059472005
Average Global Trainning Accuracy:  0.8813690568375406
Average Global Trainning Loss    :  0.4633723650810167
-------------Round number:  70  -------------
Average Test Accuracy          :  0.8827966347548593
Average Global Trainning Accuracy:  0.8821821279231842
Average Global Trainning Loss    :  0.45907141413872543
-------------Round number:  71  -------------
Average Test Accuracy          :  0.8838410211778358
Average Global Trainning Accuracy:  0.8829564813380827
Average Global Trainning Loss    :  0.4549467344669545
-------------Round number:  72  -------------
Average Test Accuracy          :  0.8842471714534378
Average Global Trainning Accuracy:  0.8839050642713334
Average Global Trainning Loss    :  0.4509077327657968
-------------Round number:  73  -------------
Average Test Accuracy          :  0.8850014505366985
Average Global Trainning Accuracy:  0.8848536472045841
Average G

-------------Round number:  26  -------------
Average Test Accuracy          :  0.7903104148534958
Average Global Trainning Accuracy:  0.787052810902896
Average Global Trainning Loss    :  0.9802978477814774
-------------Round number:  27  -------------
Average Test Accuracy          :  0.7949521322889469
Average Global Trainning Accuracy:  0.7916021372154252
Average Global Trainning Loss    :  0.952561854503833
-------------Round number:  28  -------------
Average Test Accuracy          :  0.7995358282564549
Average Global Trainning Accuracy:  0.7958030044912499
Average Global Trainning Loss    :  0.9260242185080145
-------------Round number:  29  -------------
Average Test Accuracy          :  0.8042935886277923
Average Global Trainning Accuracy:  0.8001200247793093
Average Global Trainning Loss    :  0.9009569465550953
-------------Round number:  30  -------------
Average Test Accuracy          :  0.8089933275311866
Average Global Trainning Accuracy:  0.8050178101285427
Average Glob

-------------Round number:  66  -------------
Average Test Accuracy          :  0.8798375398897592
Average Global Trainning Accuracy:  0.879045996592845
Average Global Trainning Loss    :  0.4769913435757705
-------------Round number:  67  -------------
Average Test Accuracy          :  0.8808239048447926
Average Global Trainning Accuracy:  0.8800720148675856
Average Global Trainning Loss    :  0.4723031781158046
-------------Round number:  68  -------------
Average Test Accuracy          :  0.8813460980562808
Average Global Trainning Accuracy:  0.8808463682824841
Average Global Trainning Loss    :  0.4677147182200519
-------------Round number:  69  -------------
Average Test Accuracy          :  0.8822164200754279
Average Global Trainning Accuracy:  0.8815432863558929
Average Global Trainning Loss    :  0.4633052141208185
-------------Round number:  70  -------------
Average Test Accuracy          :  0.8828546562228025
Average Global Trainning Accuracy:  0.8822789221000464
Average Glo

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     url                   : https://www.comet.com/abdulmomen96/sophia/90c6b2efb6804e06b26bfb5389728792
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     glob_acc [410]   : (0.0777487670438062, 0.8899332753118654)
COMET INFO:     loss [13120]     : (0.15834854543209076, 2.4836485385894775)
COMET INFO:     train_acc [410]  : (0.07615765835527334, 0.8897514325538176)
COMET INFO:     train_loss [410] : (0.41935872950383307, 2.3220207659807186)
COMET INFO:   Uploads:
COMET INFO:     conda-environment-definition : 1
COMET INFO:     conda-info                   : 1
COMET INFO:     conda-specification          : 1
COMET INFO:     environment details          : 1
COMET INFO:    

In [ ]:
# Run 5

In [ ]:
%reload_ext autoreload
%autoreload 2

from comet_ml import Experiment
import h5py
import matplotlib.pyplot as plt
import numpy as np
import argparse


import importlib
import random
import os
from algorithms.server.server import Server
from algorithms.trainmodel.models import *
from utils.plot_utils import *
import torch
#torch.manual_seed(0)

def main(experiment, dataset, algorithm, model, batch_size, learning_rate, alpha, eta, L, rho, num_glob_iters,
         local_epochs, optimizer, numedges, times, commet, gpu):

    device = torch.device("cuda:{}".format(gpu) if torch.cuda.is_available() and gpu != -1 else "cpu")

    for i in range(times):
        print("---------------Running time:------------",i)

        # Generate model
        if(model == "mclr"):
            if(dataset == "human_activity"):
                model = Mclr_CrossEntropy(561,6).to(device), model
                print("Hey")
            else:
                model = Mclr_Logistic().to(device), model

        if(model == "linear_regression"):
            model = Linear_Regression(40,1).to(device), model

        if model == "logistic_regression":
            model = Logistic_Regression(300).to(device), model
        
        if model == "MLP" and dataset == "a9a":
            model = DNN( input_dim = 123, output_dim = 2).to(device), model
        if model == "MLP" and dataset == "human_activity":
            model = DNN( input_dim = 561, mid_dim = 561, output_dim = 6).to(device), model
        if model == "MLP" and dataset == "w8a":
            model = DNN( input_dim = 300, output_dim = 2).to(device), model
        if model == "MLP" and dataset == "Mnist":
            model = DNN().to(device), model
        if model == 'CNN':
            model = Net().to(device), model
        # select algorithm
        if(commet):
            experiment.set_name(dataset + "_" + algorithm + "_" + model[1] + "_" + str(batch_size) + "b_" + str(learning_rate) + "lr_" + str(alpha) + "al_" + str(eta) + "eta_" + str(L) + "L_" + str(rho) + "p_" +  str(num_glob_iters) + "ge_"+ str(local_epochs) + "le_"+ str(numedges) +"u")
        server = Server(experiment, device, dataset, algorithm, model, batch_size, learning_rate, alpha, eta,  L, num_glob_iters, local_epochs, optimizer, numedges, i)
        
        server.train()
        server.test()

        
        sophia_params = {
    "dataset": "Fashion_Mnist",
    "algorithm": "Sophia",
    "model": "CNN",
    "batch_size": 64,
    "learning_rate": 0.001,
    "alpha": (0.90, 0.95),
    "eta": 1.0 * 20,
    "L": 0.1,
    "rho": 20,
    "num_glob_iters": 2 * 41,
    "local_epochs": 1,
    "optimizer": "Sophia",
    "numedges": 20,
    "times": 1,
    "commet": False,
    "gpu": 0
}



DONE_params = {
    "dataset": "Fashion_Mnist",
    "algorithm": "DONE",
    "model": "CNN",
    "batch_size": 0,
    "learning_rate": 1,
    "alpha": 0.003,
    "eta": 1.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 41,
    "local_epochs": 40,
    "optimizer": "DONE",
    "numedges": 32,
    "times": 1,
    "commet": False,
    "gpu": 0
}

GD_params = {
    "dataset": "Fashion_Mnist",
    "algorithm": "FedAvg",
    "model": "CNN",
    "batch_size": 64,
    "learning_rate": 0.01,
    "alpha": 0.03,
    "eta": 20.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 2 * 41,
    "local_epochs": 1,
    "optimizer": "DONE",
    "numedges": 20,
    "times": 1,
    "commet": False,
    "gpu": 0
}

'''
Newton_params = {
    "dataset": "Mnist",
    "algorithm": "Newton",
    "model": "MLP",
    "batch_size": 0,
    "learning_rate": 1,
    "alpha": 0.03,
    "eta": 1.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 500,
    "local_epochs": 40,
    "optimizer": "Newton",
    "numedges": 30,
    "times": 1,
    "commet": True,
    "gpu": 0
}

'''

In [ ]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **GD_params)
experiment.end()    


In [ ]:
%reload_ext autoreload
%autoreload 2

from comet_ml import Experiment
import h5py
import matplotlib.pyplot as plt
import numpy as np
import argparse


import importlib
import random
import os
from algorithms.server.server import Server
from algorithms.trainmodel.models import *
from utils.plot_utils import *
import torch
#torch.manual_seed(0)

def main(experiment, dataset, algorithm, model, batch_size, learning_rate, alpha, eta, L, rho, num_glob_iters,
         local_epochs, optimizer, numedges, times, commet, gpu):

    device = torch.device("cuda:{}".format(gpu) if torch.cuda.is_available() and gpu != -1 else "cpu")

    for i in range(times):
        print("---------------Running time:------------",i)

        # Generate model
        if(model == "mclr"):
            if(dataset == "human_activity"):
                model = Mclr_CrossEntropy(561,6).to(device), model
                print("Hey")
            else:
                model = Mclr_Logistic().to(device), model

        if(model == "linear_regression"):
            model = Linear_Regression(40,1).to(device), model

        if model == "logistic_regression":
            model = Logistic_Regression(300).to(device), model
        
        if model == "MLP" and dataset == "a9a":
            model = DNN( input_dim = 123, output_dim = 2).to(device), model
        if model == "MLP" and dataset == "human_activity":
            model = DNN( input_dim = 561, mid_dim = 561, output_dim = 6).to(device), model
        if model == "MLP" and dataset == "w8a":
            model = DNN( input_dim = 300, output_dim = 2).to(device), model
        if model == "MLP" and dataset == "Mnist":
            model = DNN().to(device), model
        if model == 'CNN':
            model = Net().to(device), model
            
        # select algorithm
        if(commet):
            experiment.set_name(dataset + "_" + algorithm + "_" + model[1] + "_" + str(batch_size) + "b_" + str(learning_rate) + "lr_" + str(alpha) + "al_" + str(eta) + "eta_" + str(L) + "L_" + str(rho) + "p_" +  str(num_glob_iters) + "ge_"+ str(local_epochs) + "le_"+ str(numedges) +"u")
        server = Server(experiment, device, dataset, algorithm, model, batch_size, learning_rate, alpha, eta,  L, num_glob_iters, local_epochs, optimizer, numedges, i)
        
        server.train()
        server.test()

        
sophia_params = {
    "dataset": "Cifar10",
    "algorithm": "Sophia",
    "model": "CNN",
    "batch_size": 64,
    "learning_rate": 0.001,
    "alpha": (0.90, 0.95),
    "eta": 1.0 * 20,
    "L": 0.1,
    "rho": 20,
    "num_glob_iters": 300,
    "local_epochs": 1,
    "optimizer": "Sophia",
    "numedges": 20,
    "times": 5,
    "commet": False,
    "gpu": 0
}



DONE_params = {
    "dataset": "Fashion_Mnist",
    "algorithm": "DONE",
    "model": "CNN",
    "batch_size": 0,
    "learning_rate": 1,
    "alpha": 0.003,
    "eta": 1.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 2 * 41,
    "local_epochs": 40,
    "optimizer": "DONE",
    "numedges": 32,
    "times": 1,
    "commet": False,
    "gpu": 0
}

GD_params = {
    "dataset": "Cifar10",
    "algorithm": "FedAvg",
    "model": "CNN",
    "batch_size": 64,
    "learning_rate": 0.01,
    "alpha": 0.03,
    "eta": 20.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 300,
    "local_epochs": 10,
    "optimizer": "DONE",
    "numedges": 20,
    "times": 5,
    "commet": False,
    "gpu": -1
}

In [ ]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **sophia_params)
experiment.end()  

In [ ]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **GD_params)
experiment.end()    

In [ ]:
'''
Human Activity
Sophia
-------------Round number:  40  -------------
Average Test Accuracy          :  0.8502321981424149
Average Global Trainning Accuracy:  0.8495139338950097
Average Global Trainning Loss    :  1.0823885845349968
Clipping rate = 0.576000928907216

"learning_rate": 0.0001,
    "alpha": (0.9, 0.95),
    "eta": 1.0 * 50,
    "L": 0.001,
    "rho": 20,


GD
-------------Round number:  40  -------------
Average Global Accuracy          :  0.20936532507739938
Average Global Trainning Accuracy:  0.20764744005184704
Average Global Trainning Loss    :  1.7220779072828905


'''

'''
MNIST MLP
Sophia 
-------------Round number:  39  -------------
Average Global Accuracy          :  0.8934867045115028
Average Global Trainning Accuracy:  0.8913894080996885
Average Global Trainning Loss    :  0.3557637022975078

GD
-------------Round number:  40  -------------
Average Global Accuracy          :  0.5679713175978488
Average Global Trainning Accuracy:  0.5671775700934579
Average Global Trainning Loss    :  1.9333872663551401

'''

In [ ]:
from utils.plot_utils import get_training_data_value, simple_read_data

In [ ]:


rs_train_acc_sop, rs_train_loss_sop, rs_glob_acc_sop = simple_read_data("Mnist_Sophia_0.01_(0.9, 0.95)_5.0_0.1_32u_64b_1_0")
rs_train_acc_done, rs_train_loss_done, rs_glob_acc_done = simple_read_data("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_80_0")
#("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_40_0")
rs_train_acc_gd, rs_train_loss_gd, rs_glob_acc_gd = simple_read_data("Mnist_FedAvg_0.008_0.03_20.0_0.001_32u_64b_10_0")


import matplotlib.pyplot as plt
import numpy as np

# Create a figure with two subplots
plt.figure(figsize=(10, 5))
x = np.arange(0, 82)
x_DONE = np.arange(0, 2 * 41, 2)
xlim = 41
print(rs_glob_acc_sop[:80].shape)
# Subplot 1: Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(x[:80], rs_glob_acc_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
plt.plot(x_DONE, rs_glob_acc_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_glob_acc_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Test Accuracy')
plt.xlabel('Communication Rounds')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Training Loss
plt.subplot(1, 2, 2)
plt.plot(x[:80], rs_train_loss_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
plt.plot(x_DONE, rs_train_loss_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_train_loss_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Training Loss')
plt.xlabel('Communication Rounds')
plt.ylabel('Loss')
plt.legend()

plt.savefig('Mnist_MLP_.eps', format='eps')

plt.tight_layout()
plt.show()


In [ ]:


rs_train_acc_sop, rs_train_loss_sop, rs_glob_acc_sop = simple_read_data("Fashion_Mnist_Sophia_0.001_(0.9, 0.95)_20.0_0.1_20u_64b_1_0_IID")
#rs_train_acc_done, rs_train_loss_done, rs_glob_acc_done = simple_read_data("Mnist_DONE_1_0.003_1.0_0.001_32u_0b_40_0")
#("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_40_0")
rs_train_acc_gd, rs_train_loss_gd, rs_glob_acc_gd = simple_read_data("Fashion_Mnist_FedAvg_0.01_0.03_20.0_0.001_20u_64b_1_0_IID")

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Create a figure with two subplots
plt.figure(figsize=(10, 5))
x = np.arange(0, 82)
x_DONE = np.arange(0, 2 * 41, 2)
xlim = 41
print(rs_glob_acc_sop[:80].shape)
# Subplot 1: Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(x[:80], rs_glob_acc_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_glob_acc_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_glob_acc_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Test Accuracy')
plt.xlabel('Communication Rounds')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Training Loss
plt.subplot(1, 2, 2)
plt.plot(x[:80], rs_train_loss_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_train_loss_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_train_loss_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Training Loss')
plt.xlabel('Communication Rounds')
plt.ylabel('Loss')
plt.legend()

plt.savefig('Mnist_MLP_.eps', format='eps')

plt.tight_layout()
plt.show()


In [ ]:


rs_train_acc_sop, rs_train_loss_sop, rs_glob_acc_sop = simple_read_data("Fashion_Mnist_Sophia_0.001_(0.9, 0.95)_20.0_0.1_20u_64b_1_0_NIID")
#rs_train_acc_done, rs_train_loss_done, rs_glob_acc_done = simple_read_data("Mnist_DONE_1_0.003_1.0_0.001_32u_0b_40_0")
#("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_40_0")
rs_train_acc_gd, rs_train_loss_gd, rs_glob_acc_gd = simple_read_data("Fashion_Mnist_GD_0.01_0.03_20.0_0.001_20u_64b_1_0_NIID")

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Create a figure with two subplots
plt.figure(figsize=(10, 5))
x = np.arange(0, 82)
x_DONE = np.arange(0, 2 * 41, 2)
xlim = 41
print(rs_glob_acc_sop[:80].shape)
# Subplot 1: Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(x[:80], rs_glob_acc_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_glob_acc_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_glob_acc_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Test Accuracy')
plt.xlabel('Communication Rounds')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Training Loss
plt.subplot(1, 2, 2)
plt.plot(x[:80], rs_train_loss_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_train_loss_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_train_loss_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Training Loss')
plt.xlabel('Communication Rounds')
plt.ylabel('Loss')
plt.legend()

plt.savefig('Mnist_MLP_.eps', format='eps')

plt.tight_layout()
plt.show()


In [ ]:
from utils.plot_utils import get_training_data_value, simple_read_data

rs_train_acc_sop, rs_train_loss_sop, rs_glob_acc_sop = simple_read_data("Cifar10_Sophia_0.001_(0.9, 0.95)_20.0_0.1_20u_64b_1_0")
#rs_train_acc_done, rs_train_loss_done, rs_glob_acc_done = simple_read_data("Mnist_DONE_1_0.003_1.0_0.001_32u_0b_40_0")
#("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_40_0")
rs_train_acc_gd, rs_train_loss_gd, rs_glob_acc_gd = simple_read_data("Cifar10_FedAvg_0.01_0.03_20.0_0.001_20u_64b_10_0")


import matplotlib.pyplot as plt
import numpy as np

# Create a figure with two subplots
plt.figure(figsize=(10, 5))
x = np.arange(0, 300)
x_DONE = np.arange(0, 2 * 41, 2)
xlim = 41
print(rs_glob_acc_sop[:80].shape)
# Subplot 1: Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(x[:299], rs_glob_acc_sop[0:299], label='FedSophia', linestyle='--', marker='o', markevery=20)
#plt.plot(x_DONE, rs_glob_acc_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x[:299], rs_glob_acc_gd[0:299], label='FedAVG', linestyle='--', marker='D', markevery=20)

plt.title('Test Accuracy')
plt.xlabel('Communication Rounds')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Training Loss
plt.subplot(1, 2, 2)
plt.plot(x[:299], rs_train_loss_sop[0:299], label='FedSophia', linestyle='--', marker='o', markevery=20)
#plt.plot(x_DONE, rs_train_loss_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x[:299], rs_train_loss_gd[0:299], label='FedAVG', linestyle='--', marker='D', markevery=20)

plt.title('Training Loss')
plt.xlabel('Communication Rounds')
plt.ylabel('Loss')
plt.legend()

plt.savefig('Mnist_MLP_.eps', format='eps')

plt.tight_layout()
plt.show()


In [ ]:
from utils.plot_utils import get_training_data_value, simple_read_data

rs_train_acc_sop, rs_train_loss_sop, rs_glob_acc_sop = simple_read_data("Cifar10_Sophia_0.001_(0.9, 0.95)_20.0_0.1_20u_64b_1_0_NIID")
#rs_train_acc_done, rs_train_loss_done, rs_glob_acc_done = simple_read_data("Mnist_DONE_1_0.003_1.0_0.001_32u_0b_40_0")
#("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_40_0")
rs_train_acc_gd, rs_train_loss_gd, rs_glob_acc_gd = simple_read_data("Cifar10_FedAvg_0.01_0.03_20.0_0.001_20u_64b_10_0_NIID")


import matplotlib.pyplot as plt
import numpy as np

# Create a figure with two subplots
plt.figure(figsize=(10, 5))
x = np.arange(0, 300)
x_DONE = np.arange(0, 2 * 41, 2)
xlim = 41
print(rs_glob_acc_sop[:80].shape)
# Subplot 1: Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(x[:299], rs_glob_acc_sop[0:299], label='FedSophia', linestyle='--', marker='o', markevery=20)
#plt.plot(x_DONE, rs_glob_acc_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x[:299], rs_glob_acc_gd[0:299], label='FedAVG', linestyle='--', marker='D', markevery=20)

plt.title('Test Accuracy')
plt.xlabel('Communication Rounds')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Training Loss
plt.subplot(1, 2, 2)
plt.plot(x[:299], rs_train_loss_sop[0:299], label='FedSophia', linestyle='--', marker='o', markevery=20)
#plt.plot(x_DONE, rs_train_loss_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x[:299], rs_train_loss_gd[0:299], label='FedAVG', linestyle='--', marker='D', markevery=20)

plt.title('Training Loss')
plt.xlabel('Communication Rounds')
plt.ylabel('Loss')
plt.legend()

plt.savefig('Mnist_MLP_.eps', format='eps')

plt.tight_layout()
plt.show()


In [ ]:
#1.027

np.mean(np.equal(np.ones((20, 1), np.float32), 0.0))

In [ ]:
model = DNN( input_dim = 123, output_dim = 2)


In [ ]:

        #print(grads[i].sign())
#print(f"Clipping rate = {1 - (winrate / param_count)}")
x = sum(p.numel() for p in model.parameters() if p.requires_grad)
x 

In [ ]:
"""
Mnist MLP settings
sophia_params = {
    "dataset": "Mnist",
    "algorithm": "Sophia",
    "model": "MLP",
    "batch_size": 64,
    "learning_rate": 0.01,
    "alpha": (0.90, 0.95),
    "eta": 1.0 * 20,
    "L": 0.1,
    "rho": 20,
    "num_glob_iters": 2 * 41,
    "local_epochs": 1,
    "optimizer": "Sophia",
    "numedges": 32,
    "times": 1,
    "commet": False,
    "gpu": 0
}



DONE_params = {
    "dataset": "Mnist",
    "algorithm": "DONE",
    "model": "MLP",
    "batch_size": 64,
    "learning_rate": 1,
    "alpha": 0.003,
    "eta": 1.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 41,
    "local_epochs": 40,
    "optimizer": "DONE",
    "numedges": 32,
    "times": 1,
    "commet": True,
    "gpu": 0
}

GD_params = {
    "dataset": "Mnist",
    "algorithm": "GD",
    "model": "MLP",
    "batch_size": 64,
    "learning_rate": 0.01,
    "alpha": 0.03,
    "eta": 20.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 2 * 41,
    "local_epochs": 1,
    "optimizer": "DONE",
    "numedges": 30,
    "times": 1,
    "commet": False,
    "gpu": 0
}







"""

In [ ]:
class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 2, 1)
        self.conv2 = nn.Conv2d(16, 32, 2, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(18432, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = nn.ReLU()(x)
        x = nn.MaxPool2d(2, 1)(x)
        x = self.dropout1(x)
        x = self.conv2(x)
        x = nn.ReLU()(x)
        x = nn.MaxPool2d(2, 1)(x)
        x = self.dropout2(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = nn.ReLU()(x)
        x = self.fc2(x)
        output = x#F.log_softmax(x, dim=1)
        return output
    


In [ ]:
model = Net()

In [ ]:
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of parameters: {param_count}")

In [ ]:
class CNN28(nn.Module):
    def __init__(self):
        super(CNN28, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)  # Adjusted input size based on the dimensions after pooling
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(nn.ReLU()(self.conv1(x)))
        x = self.pool(nn.ReLU()(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = nn.ReLU()(self.fc1(x))
        x = self.fc2(x)
        return x

# Create an instance of the model
model = CNN28()

In [ ]:
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of parameters: {param_count}")